## Data Quality Checks and Audits

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path("../data")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

raw = pd.read_csv(DATA / "renewals_raw_extract.csv")

In [ ]:
print("Shape : ", raw.shape)
print(raw.dtypes.astype(str).value_counts().to_dict())
print("Colums that are read as text that should be Numbers or Dates ? : ",
      [column for column in ["renewal_date", "auto_renew"] if raw[column].dtype != "bool"])

In [ ]:
raw.head()

In [ ]:
raw.sample(5, random_state=1)

### Missing Values

In [ ]:
miss = raw.isna().sum()

miss = pd.DataFrame({
    "missing": miss,
    "pct": (miss / len(raw) * 100).round(1)
})

print(miss[miss["missing"] > 0].sort_values("missing", ascending=False))

### Duplicate Values

In [ ]:
print("Exact Duplicate Rows :", raw.duplicated().sum())
print("Duplicate Subscription Ids : ", raw["subscription_id"].duplicated().sum())

key = raw["subscription_id"].str.strip().str.upper()
print("Duplicated Subscription Ids :", key.duplicated().sum())

odd_ids = raw.loc[~raw["subscription_id"].str.fullmatch(r"SUB-\d{5}"), "subscription_id"]
print("IDs not matching the Pattern ...", odd_ids.tolist())

### Inconsistent Categories

In [ ]:
for column in ["region", "segment", "tier", "product", "auto_renew"]:
    print(f"{column}: {raw[column].value_counts(dropna=False).to_dict()}")

### Data related Quality Checks

In [ ]:
iso = pd.to_datetime(
    raw["renewal_date"], format="%Y-%m-%d", errors="coerce"
)

print("Parse as YYY-MM-DD : ", iso.notna().sum(),
      " Ok : ", iso.isna().sum(), "fail")

### Valid Ranges

In [ ]:
key_columns = ["seats_purchased", "discount_pct", "annual_contract_value",
               "avg_seat_utilization", "nps_score"]

print(raw[key_columns].agg(["min", "max", "median"]).T)

rules = {
    "seats < 0 or >= 50000": (raw["seats_purchased"] < 0) | (raw["seats_purchased"] >= 50000),
    "discount outside 0-40%": (raw["discount_pct"] < 0) | (raw["discount_pct"] > 40),
    "ACV <= 0": raw["annual_contract_value"] <= 0,
    "utilization outside 0-1": (raw["avg_seat_utilization"] < 0) | (raw["avg_seat_utilization"] > 1),
    "NPS outside -100 to 100": (raw["nps_score"] < -100) | (raw["nps_score"] > 100),
}

print()
for rule, broken in rules.items():
    print(f"{rule:<24}: {broken.sum()} rows")

### Data Quality Scorecard

In [ ]:
issues = {
    "missing cells": int(raw.drop(columns=["renewal_acv", "churn_reason", "csm_health", "last_csm_note"]).isna().sum().sum()),
    "duplicate rows by cleaned id": int(key.duplicated().sum()),
    "rows breaking a range rule": int(pd.concat(rules, axis=1).any(axis=1).sum()),
    "dates not in ISO format": int(iso.isna().sum())
}

for k, v in issues.items():
    print(f"{k} - {v}")